In [2]:
result1=[]
result1.append({
        "動画本数": len(df),
        "平均再生回数": df["views"].mean(),
        "再生数の中央値": df["views"].median(),
        "平均高評価数": df["likes"].mean(),
        "平均高評価率": df["like_rate"].mean(),
        "平均コメント率": df["comment_rate"].mean(),
})
df_a=pd.DataFrame(result1)
df_a


,動画本数,平均再生回数,再生数の中央値,平均高評価数,平均高評価率,平均コメント率
0,367,7106.207084,626.0,123.316076,2.874744,0.847828


In [1]:
import requests
import pandas as pd
API_KEY= ''

search_url= "https://www.googleapis.com/youtube/v3/search"

search_params={
    "part":"snippet",
    "q": "가와고에|카와고에",
    "videoCategoryId":19,
    "type":"video",
    "key":API_KEY,
    "relevanceLanguage":"ko",
    "regionCode": "KR",
    "maxResults":50

}
page_token=None
videos=[]
for i in range(10):
 if page_token:
        search_params["pageToken"] = page_token
 response=requests.get(search_url,params=search_params)
 data=response.json()

 video_ids=[]

 for video in data["items"]:
  video_id=video["id"]["videoId"]
  video_ids.append(video_id)

 video_url = "https://www.googleapis.com/youtube/v3/videos"

 video_params={
    "part":"snippet,statistics",
    "id":",".join(video_ids),
    "key":API_KEY
}

 response=requests.get(video_url,params=video_params)
 data_videos=response.json()



 for video in data_videos["items"]:
  video_id=video["id"]

  title=video["snippet"]["title"]
  published_at = video["snippet"]["publishedAt"]
  description= video["snippet"]["description"]
  tags1=video["snippet"].get("tags",[])
  tags=",".join(tags1)

  views = int(video["statistics"].get("viewCount",0))
  likes = int(video["statistics"].get("likeCount",0))
  comments = int(video["statistics"].get("commentCount",0))

  videos.append({
        "video_id": video_id,
        "title": title,
        "description":description,
        "tags":tags,
        "published_at": published_at,
        "views": views,
        "likes": likes,
        "comments": comments
    })
 page_token = data.get("nextPageToken")


df = pd.DataFrame(videos)
df = df.drop_duplicates(subset="video_id")
print("\n===== YouTube動画データ =====")
df["like_rate"] = df["likes"] / df["views"] * 100
df["comment_rate"] = df["comments"] / df["views"] * 100
print(df)



===== YouTube動画データ =====
        video_id                                              title  \
0    Ej3DYRsI4_0  요즘 뜨는 도쿄근교여행 | 뚜벅이로 30분만에 가볼수 있는 일본 감성 로컬 카와고에...   
1    _9ZkfKF3ORc                         에도시대 감성 가득한 가와고에(川越) 당일 여행   
2    _NWC_9_sLQQ       도쿄 근교 여행지로 딱! 반일치기 가와고에 여행 코스&맛집&명소 v-log🤍🇯🇵   
3    O1Do725qgGo      도쿄 근교 소도시 가와고에 | 작은 에도 시대로 떠나는 레트로 감성 시간 여행☎️   
4    G6R-ufrw_QI             도쿄 브이로그 | 비오는날 반일치기 근교여행 #가와고에 #작은에도마을   
..           ...                                                ...   
445  hIAW3rO3RSM    Best Japanese Street Food Tour of Kawagoe Japan   
446  9RZdjTCrIJE  MUST MAKE souvenir in Japan: Chopsticks! 🥢 #japan   
447  O93cyOdJjSQ  Ikan pembawa hoki #kawagoe #kawagoehikawa #omi...   
448  LzyKAoEvyFU      야마노테선 신주쿠, 이케부쿠로 방면, 사이쿄선 쾌속 카와고에행 열차 시부야역 진입   
449  ODyWEKkX7i8  Kawagoe, Little Edo City walking tour at Saita...   

                                           description  \
0    。 카와고에 여행가이드 구글링크\nhttps://maps.app.goo.gl/mzD...   
1    

In [ ]:
keywords_kr = {
    "日帰り": ["당일치기"],
    "グルメ": ["맛집", "미식가"],
    "川越": ["가와고에", "카와고에"],
    "江戸": ["에도"],
    "神社": ["신사"],
    "サツマイモ": ["고구마"],
    "着物":["기모노"],
    "スイーツ":["디저트","후식","스위츠"],
    "お菓子":["과자"],
    "東京近郊": ["도쿄 근교"],
    "観光": ["관광", "관광지", "관광 명소"],
    "食べ歩き": ["먹거리", "길거리 음식", "먹방"]
}

results = []

for keyword_jp, keywords in keywords_kr.items():

    condition = False

    for keyword in keywords:
        condition = (
            condition |
            df["title"].str.contains(keyword, na=False, regex=False) |
            df["description"].str.contains(keyword, na=False, regex=False) |
            df["tags"].str.contains(keyword, na=False, regex=False)
        )

    target = df[condition]

    results.append({
        "キーワード": keyword_jp,
        "検索語": ", ".join(keywords),
        "動画本数": len(target),

        "平均再生回数": target["views"].mean(),
        "全体との差（平均再生回数）":
            target["views"].mean() - df["views"].mean(),

        "再生数の中央値": target["views"].median(),
        "全体との差（再生数の中央値）":
            target["views"].median() - df["views"].median(),

        "平均高評価数": target["likes"].mean(),
        "全体との差（平均高評価数）":
            target["likes"].mean() - df["likes"].mean(),

        "平均高評価率": target["like_rate"].mean(),
        "全体との差（平均高評価率）":
            target["like_rate"].mean() - df["like_rate"].mean(),

        "平均コメント率": target["comment_rate"].mean(),
        "全体との差（平均コメント率）":
            target["comment_rate"].mean() - df["comment_rate"].mean()
    })

keyword_df = pd.DataFrame(results)

keyword_df = keyword_df.sort_values(
    "平均再生回数",
    ascending=False
)

keyword_df

,キーワード,検索語,動画本数,平均再生回数,全体との差（平均再生回数）,再生数の中央値,全体との差（再生数の中央値）,平均高評価数,全体との差（平均高評価数）,平均高評価率,全体との差（平均高評価率）,平均コメント率,全体との差（平均コメント率）
5,サツマイモ,고구마,35,8194.114286,1093.905330,569.0,-22.0,77.028571,-47.783795,2.266040,-0.570096,0.868261,0.060477
1,グルメ,"맛집, 미식가",60,7397.250000,297.041045,549.5,-41.5,144.283333,19.470967,2.459132,-0.377004,0.731450,-0.076333
3,江戸,에도,113,7104.185841,3.976885,494.0,-97.0,103.929204,-20.883163,3.543077,0.706942,1.693774,0.885990
9,東京近郊,도쿄 근교,66,6426.772727,-673.436228,643.0,52.0,94.166667,-30.645700,2.505731,-0.330405,0.594101,-0.213683
0,日帰り,당일치기,37,6058.756757,-1041.452198,780.0,189.0,118.972973,-5.839394,2.289661,-0.546474,0.503582,-0.304202
6,着物,기모노,6,5018.500000,-2081.708955,875.5,284.5,131.333333,6.520967,2.009494,-0.826641,0.621792,-0.185991
4,神社,신사,51,4833.843137,-2266.365818,569.0,-22.0,45.333333,-79.479033,3.013886,0.177751,2.035057,1.227274
2,川越,"가와고에, 카와고에",271,4819.712177,-2280.496778,484.0,-107.0,60.000000,-64.812367,3.016052,0.179916,1.257592,0.449808
8,お菓子,과자,25,4121.520000,-2978.688955,549.0,-42.0,62.200000,-62.612367,2.139952,-0.696184,0.606932,-0.200852
11,食べ歩き,"먹거리, 길거리 음식, 먹방",33,2029.545455,-5070.663501,645.0,54.0,39.333333,-85.479033,2.242235,-0.593900,0.684273,-0.123510


In [ ]:
result1=[]
result1.append({
        "動画本数": len(df),
        "平均再生回数": df["views"].mean(),
        "再生数の中央値": df["views"].median(),
        "平均高評価数": df["likes"].mean(),
        "平均高評価率": df["like_rate"].mean(),
        "平均コメント率": df["comment_rate"].mean(),
})
df_a=pd.DataFrame(result1)
df_a


,動画本数,平均再生回数,再生数の中央値,平均高評価数,平均高評価率,平均コメント率
0,367,7106.207084,626.0,123.316076,2.874744,0.847828


In [ ]:
keyword_df.to_csv("kawagoe_ko.csv", index=False, encoding="utf-8-sig")